In [ ]:
import os
import time
import csv
import json
import requests
from datetime import datetime


import telebot
from dotenv import load_dotenv

load_dotenv(override=False)
TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")

if not TOKEN:
    raise RuntimeError(
        "Falta TELEGRAM_BOT_TOKEN. Creá el archivo .env junto al notebook."
    )

bot = telebot.TeleBot(TOKEN, parse_mode=None)

RUTA_CSV = "historial.csv"

def registrar(message):
    existe = os.path.isfile(RUTA_CSV)

    with open(RUTA_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not existe:
            writer.writerow(["timestamp", "chat_id", "nombre", "mensaje"])

        writer.writerow([
            datetime.now().isoformat(timespec="seconds"),
            message.chat.id,
            message.from_user.first_name or "sin nombre",
            message.text or f"[{message.content_type}]",
        ])

print("Bot creado en Python. Todavía no está escuchando mensajes.")

URL_DOLAR_ACTUAL = "https://api.bluelytics.com.ar/v2/latest"

headers = {
    "User-Agent": "TDI2-BotRecetas/1.0 (trabajo practico; consultas de bajo volumen)"
}


def pedir_json(url, params=None, timeout=20):
    try:
        respuesta = requests.get(url, params=params, headers=headers timeout=timeout)
        respuesta.raise_for_status()
        return respuesta.json()

    except requests.exceptions.Timeout:
        print("La API tardó demasiado. Probá más tarde.")
    except requests.exceptions.ConnectionError:
        print("No se pudo conectar. Revisá la conexión o el estado de la API.")
    except requests.exceptions.HTTPError as error:
        print(f"La API respondió un error HTTP: {error}")
    except requests.exceptions.JSONDecodeError:
        print("La respuesta no era JSON válido.")

    return None

Bot creado en Python. Todavía no está escuchando mensajes.


In [2]:
@bot.message_handler(commands=["start"])
def comando_start(message):
    registrar(message)
    nombre = message.from_user.first_name or "persona"
    texto = (
        f"¡Hola, {nombre}! 👋, soy un bot de ejemplo creado en Python 👾. Me has iniciado 🤖🦾"
    )
    foto = open("imagenes/foto_bot_cocinando.jpeg", "rb")
    bot.send_photo(message.chat.id, foto, caption= "Estoy cocinando mis comandos para vos 🥘")
    foto.close()

@bot.message_handler(commands=["atajos"])
def comando_atajos(message):
    teclado = types.InlineKeyboardMarkup(row_width=2)

    btn_start = types.InlineKeyboardButton("/start", callback_data="start")
    btn_dolar = types.InlineKeyboardButton("/dolar", callback_data="dolar")
    btn_ingredientes = types.InlineKeyboardButton("/ingredientes", callback_data="ingredientes")
    btn_historial = types.InlineKeyboardButton("/historial", callback_data="historial")
    btn_receta = types.InlineKeyboardButton("/receta", callback_data="receta")
    
    teclado.add(btn_start, btn_dolar, btn_ingredientes, btn_historial, btn_receta)
    
    texto = "*Podes usar estos comandos rápidos:*"
    
    bot.reply_to(message, texto, parse_mode="HTML", reply_markup=teclado)

@bot.message_handler(commands=["ingredientes"])
def comando_ingredientes(message):
    texto = ("Esto todavia no fue implementado")
    bot.reply_to(message, texto)

@bot.message_handler(commands=["dolar"])
def comando_dolar(message):
    registrar(message)

    dolar_actual = pedir_json(URL_DOLAR_ACTUAL)

    if dolar_actual is not None:
        bot.reply_to(message, "Dólar blue: $" + str(dolar_actual["blue"]["value_sell"]))
    else:
        bot.reply_to(message, "No pude consultar la cotización 😕")
        

@bot.message_handler(commands=["historial"])
def comando_historial(message):
    registrar(message)
    chat_id = message.chat.id

    if not os.path.isfile(RUTA_CSV):
        bot.reply_to(message, "Todavía no hay historial 📭")
        return

    with open(RUTA_CSV, "r", encoding="utf-8") as f:
        filas = list(csv.DictReader(f))

    mias = [f for f in filas if f["chat_id"] == str(chat_id)]
    ultimas = mias[-5:]

    if not ultimas:
        bot.reply_to(message, "No encontré consultas tuyas todavía 📭")
        return

    lineas = [f"• {f['timestamp'][11:16]} — {f['mensaje']}" for f in ultimas]
    texto = "🕐 <b>Tus últimas 5 consultas:</b>\n\n" + "\n".join(lineas)

    bot.send_message(chat_id, texto, parse_mode="HTML")
    
@bot.message_handler(commands=["receta"])
def comando_receta(message):
    registrar(message)
    texto = ("esto todavia no fue implementado")
    bot.reply_to(message, "<b>Buscando una receta para vos!</b> 👨‍🍳", parse_mode = "HTML")
    bot.send_message(message.chat.id, texto)


In [3]:
@bot.message_handler(content_types=["text", "document", "audio"])
def mensaje_de_texto(message):
    if message.content_type == "document":
        bot.reply_to(message, "¡Recibí tu documento! Gracias por enviarlo.")
        return
    elif message.content_type == "audio":
        bot.reply_to(message, "¡Qué buen audio! Lo recibí correctamente.")
        return

    texto = (message.text or "").strip()
    bot.send_chat_action(message.chat.id, "typing")
    
    if texto.startswith("/"):
        bot.reply_to(message, "No conozco ese comando. Probá con /atajos.")
        return
        
    bot.reply_to(message, "¡Hola! Podes ver los comandos en el menú o usando /atajos")

In [4]:
from telebot import types

bot.set_my_commands([
    types.BotCommand("start", "Iniciar la conversación"),
    types.BotCommand("atajos", "Ver atajos"),
    types.BotCommand("ingredientes", "Ver ingredientes"),
    types.BotCommand("dolar", "Ver cotización"),
    types.BotCommand("historial", "Ver historial"),
    types.BotCommand("receta", "Pedir recetas")
])

print("Menú de comandos configurado.")


Menú de comandos configurado.


In [5]:
# Ejecutar esta celda al final. Mantener el notebook abierto mientras se prueba el bot.
bot.remove_webhook()
time.sleep(1)
print("Bot escuchando con polling. Para detenerlo: Interrupt Kernel.")
bot.infinity_polling(
    skip_pending=True,
    timeout=20,
    long_polling_timeout=20,
    allowed_updates=["message"],
)

Bot escuchando con polling. Para detenerlo: Interrupt Kernel.


2026-09-29 12:51:41,908 (__init__.py:1160 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2026-09-29 12:51:41,909 (__init__.py:1162 MainThread) ERROR - TeleBot: "Break infinity polling"
